**https://www.geeksforgeeks.org/python/copy-python-deep-copy-shallow-copy/**

# Copy vs Deepcopy — Simple Backend Interview Understanding

# 1. First Understand the REAL Problem

In Python, variables mostly store **references**, not actual objects.

```python
a = {"name": "Vishal"}
b = a
```

Now both `a` and `b` point to SAME object.

So:

```python
b["name"] = "Rahul"

print(a)
```

Output:

```python
{'name': 'Rahul'}
```

Because both are same object in memory.

This causes many backend bugs.

---

# 2. Why Copy Exists

Suppose in backend service:

```python
request_data = {
    "user": "vishal",
    "permissions": ["read"]
}
```

You want to modify data safely for one request without affecting original shared object.

That is why copying exists.

---

# 3. Shallow Copy (`copy.copy`)

```python
import copy

a = {
    "user": "vishal",
    "permissions": ["read"]
}

b = copy.copy(a)
```

Now:

* outer dictionary is new
* BUT nested list is shared

So:

```python
b["permissions"].append("write")

print(a)
```

Output:

```python
{'user': 'vishal', 'permissions': ['read', 'write']}
```

Because nested list shared.

---

# Easy Memory Understanding

```text
a  ---> dict1 ---> permissions_list

b  ---> dict2 ---> same permissions_list
```

Only outer dict copied.

Inner objects shared.

---

# 4. Deepcopy (`copy.deepcopy`)

```python
import copy

a = {
    "permissions": ["read"]
}

b = copy.deepcopy(a)

b["permissions"].append("write")

print(a)
```

Output:

```python
{'permissions': ['read']}
```

Now everything copied recursively.

---

# Easy Memory Understanding

```text
a ---> dict1 ---> list1

b ---> dict2 ---> list2
```

Completely independent.

---

# 5. MOST IMPORTANT INTERVIEW DIFFERENCE

| Feature               | copy | deepcopy |
| --------------------- | ---- | -------- |
| Outer object copied   | Yes  | Yes      |
| Nested objects copied | No   | Yes      |
| Faster                | Yes  | No       |
| Safer isolation       | No   | Yes      |

---

# 6. Most Common Production Example

# Shared API Response Bug

```python
BASE_RESPONSE = {
    "status": "success",
    "meta": {}
}
```

Request handler:

```python
response = copy.copy(BASE_RESPONSE)

response["meta"]["request_id"] = "abc123"
```

Problem:

All requests share same `meta` dictionary.

Result:

* wrong logs
* mixed request IDs
* debugging nightmare

---

# Correct Fix

```python
response = copy.deepcopy(BASE_RESPONSE)
```

OR better:

```python
response = {
    "status": "success",
    "meta": {
        "request_id": request_id
    }
}
```

Senior engineers often prefer rebuilding small objects instead of deepcopy.

---

# 7. Most Common Interview Questions

# Q1. Difference between copy and deepcopy?

## Easy Strong Answer

Shallow copy creates new outer object but nested mutable objects are shared.

Deepcopy recursively copies everything and creates fully independent objects.

---

# Q2. Why can shallow copy cause bugs?

Because nested mutable objects are shared between original and copied object.

Changing one changes other.

---

# Q3. Why not always use deepcopy?

Because deepcopy is slower and uses more memory.

In high traffic backend services, unnecessary deepcopies hurt performance.

---

# Q4. When is shallow copy safe?

When nested objects are immutable or when sharing is intentional.

---

# 8. What Interviewers ACTUALLY Want

For 2 YOE, interviewer mainly checks:

✅ Do you understand references?
✅ Do you understand nested mutable object sharing?
✅ Can you predict mutation bugs?
✅ Do you know deepcopy is expensive?
✅ Can you explain one real production issue?

That is enough for most backend interviews.

---

# 9. Final Revision Notes

✅ Assignment does NOT copy object
✅ `copy.copy()` → only outer object copied
✅ `copy.deepcopy()` → everything copied recursively
✅ Nested mutable objects are main source of bugs
✅ Deepcopy safer but slower
✅ Common backend bug = shared request/cache/config mutation
